In [25]:
import random
import numpy as np
import os

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed(42)


# Stacking  XGBoost+ CatBoost+ Ridge+ LGBM

* This notebook stacks the XGBoost and LGBM and Ridge and CatBoost models.

* The reference notes for data processing are below.

https://www.kaggle.com/code/mdshahbazalam/lgbm-multioutputregressor

# If this note is useful, Please VOTE!

In [26]:
import os

In [27]:
import pandas as pd
import numpy as np
import re
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import xgboost as xgb

In [28]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression

In [29]:
import pandas as pd
train=pd.read_csv('input/feedback-prize-english-language-learning/train.csv')
display(train.head(2))

,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions
0,0016926B079C,I think that students would benefit from learn...,3.5,3.5,3.0,3.0,4.0,3.0
1,0022683E9EA5,When a problem is a change you have to let it ...,2.5,2.5,3.0,2.0,2.0,2.5


In [30]:
print('No. of rows in train datasets',train.shape[0])

No. of rows in train datasets 100


In [31]:
def data_cleaner(text):
    text = text.strip()
    text = re.sub(r'\n', '', text)
    text = text.lower()
    return text

In [32]:
train['full_text']=train['full_text'].apply(data_cleaner)

In [33]:
import nltk
from tqdm import tqdm
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
def generate_sentiment_scores(data):
    sid = SentimentIntensityAnalyzer()
    neg=[]
    pos=[]
    neu=[]
    comp=[]
    for sentence in tqdm(data['full_text'].values): 
        sentence_sentiment_score = sid.polarity_scores(sentence)
        comp.append(sentence_sentiment_score['compound'])
        neg.append(sentence_sentiment_score['neg'])
        pos.append(sentence_sentiment_score['pos'])
        neu.append(sentence_sentiment_score['neu'])
    return comp,neg,pos,neu
train['compound'],train['negative'],train['positive'],train['neutral']=generate_sentiment_scores(train)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\alexa\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
100%|██████████| 100/100 [00:00<00:00, 364.51it/s]


In [34]:
train['com_len']=train['full_text'].apply(lambda x:len(x.split()))

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train['full_text'])

In [36]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_com=trans.fit_transform(train['compound'].values.reshape(-1,1))

CPU times: total: 15.6 ms
Wall time: 1 ms


In [37]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_neg=trans.fit_transform(train['negative'].values.reshape(-1,1))

CPU times: total: 0 ns
Wall time: 1 ms


In [38]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_pos=trans.fit_transform(train['positive'].values.reshape(-1,1))

CPU times: total: 0 ns
Wall time: 1e+03 μs


In [39]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_neu=trans.fit_transform(train['neutral'].values.reshape(-1,1))

CPU times: total: 0 ns
Wall time: 1.16 ms


In [40]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_len=trans.fit_transform(train['com_len'].values.reshape(-1,1))

CPU times: total: 0 ns
Wall time: 505 μs


In [41]:
%%time
from scipy.sparse import hstack
train_s=hstack((X_train,X_train_com,X_train_neg,X_train_pos,X_train_neu,X_train_len))

CPU times: total: 0 ns
Wall time: 2 ms


In [42]:
y=train[['cohesion','syntax','vocabulary','phraseology','grammar','conventions']]

In [43]:
print(train_s.shape,y.shape)

(100, 3177) (100, 6)


In [44]:
params_lgb = {
    "n_estimators": 1000,
    "verbose": -1,
    "random_state": 42
}

In [45]:
y_train=train[['cohesion','syntax','vocabulary','phraseology','grammar','conventions']]

In [46]:
model = MultiOutputRegressor(LGBMRegressor(**params_lgb))
model.fit(train_s, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,"LGBMRegressor...2, verbose=-1)"
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [47]:
param = {'learning_rate': 0.3, 
          'depth': 12, 
          'l2_leaf_reg': 4, 
          'loss_function': 'MultiRMSE', 
          'eval_metric': 'MultiRMSE', 
          'task_type': 'CPU', 
          'iterations': 20,
          'od_type': 'Iter', 
          'boosting_type': 'Plain', 
          'bootstrap_type': 'Bayesian', 
          'allow_const_label': True, 
          'random_state': 42,
          'allow_writing_files': False
         }

In [48]:
model2 = CatBoostRegressor(**param)
model2.fit(train_s, y_train)

0:	learn: 1.6892226	total: 8.96ms	remaining: 170ms
1:	learn: 1.6080667	total: 8.96ms	remaining: 170ms
2:	learn: 1.5086441	total: 8.96ms	remaining: 170ms
3:	learn: 1.4315030	total: 8.96ms	remaining: 170ms
4:	learn: 1.3876586	total: 240ms	remaining: 1.8s
5:	learn: 1.3088624	total: 2.25s	remaining: 10.5s
6:	learn: 1.2390611	total: 4.25s	remaining: 13.8s
7:	learn: 1.1641007	total: 6.4s	remaining: 15.4s
8:	learn: 1.1048729	total: 8.44s	remaining: 15.5s
9:	learn: 1.0455559	total: 10.5s	remaining: 14.9s
10:	learn: 1.0084534	total: 12.5s	remaining: 14s
11:	learn: 0.9610228	total: 14.5s	remaining: 12.9s
12:	learn: 0.9107678	total: 15.5s	remaining: 10.8s
13:	learn: 0.8570164	total: 17.5s	remaining: 9.55s
14:	learn: 0.8442612	total: 17.6s	remaining: 7.33s
15:	learn: 0.7983763	total: 19.6s	remaining: 6.04s
16:	learn: 0.7641296	total: 19.7s	remaining: 4.22s
17:	learn: 0.7321480	total: 21.7s	remaining: 2.89s
18:	learn: 0.7016491	total: 22.7s	remaining: 1.42s
19:	learn: 0.6605278	total: 24.7s	remaini

In [49]:
model3 = Ridge(copy_X=False, random_state=42)
model3.fit(train_s, y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",False
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",42


In [50]:
xgb_estimator = xgb.XGBRegressor(
        n_estimators=500, random_state=42, 
        objective='reg:squarederror')

# create MultiOutputClassifier instance with XGBoost model inside
model4 = MultiOutputRegressor(xgb_estimator, n_jobs=2)
# model4 = XGBClassifier(early_stopping_rounds=10)
model4.fit(train_s, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,"XGBRegressor(...ree=None, ...)"
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",2
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None


In [51]:
first_pred_1 = model.predict(train_s)
first_pred_2 = model2.predict(train_s)
first_pred_3 = model3.predict(train_s)
first_pred_4 = model4.predict(train_s)

stack_pred = np.column_stack((first_pred_1,first_pred_2,first_pred_3,first_pred_4))

C:\Users\alexa\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\alexa\venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
C:\Users\alexa\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\alexa\venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
C:\Users\alexa\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\alexa\venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to

In [52]:
params = {'learning_rate': 0.3, 
          'depth': 12, 
          'l2_leaf_reg': 4, 
          'loss_function': 'MultiRMSE', 
          'eval_metric': 'MultiRMSE', 
          'task_type': 'CPU', 
          'iterations': 20,
          'od_type': 'Iter', 
          'boosting_type': 'Plain', 
          'bootstrap_type': 'Bayesian', 
          'allow_const_label': True, 
          'random_state': 42,
          'allow_writing_files': False
         }


In [53]:
# メタモデルの作成
# meta_model = LinearRegression()
meta_model =  CatBoostRegressor(**params)
# meta_model = MultiOutputRegressor(LGBMRegressor(**params_lgb))

meta_model.fit(stack_pred, y_train)

0:	learn: 1.4786777	total: 247ms	remaining: 4.69s
1:	learn: 1.3333660	total: 720ms	remaining: 6.48s
2:	learn: 1.1678260	total: 1.16s	remaining: 6.56s
3:	learn: 1.0479003	total: 1.52s	remaining: 6.09s
4:	learn: 0.9421911	total: 1.89s	remaining: 5.67s
5:	learn: 0.8534756	total: 2.26s	remaining: 5.27s
6:	learn: 0.7764701	total: 2.64s	remaining: 4.91s
7:	learn: 0.7144928	total: 2.99s	remaining: 4.49s
8:	learn: 0.6549840	total: 3.35s	remaining: 4.09s
9:	learn: 0.6065236	total: 3.72s	remaining: 3.72s
10:	learn: 0.5565502	total: 4.08s	remaining: 3.34s
11:	learn: 0.5168278	total: 4.44s	remaining: 2.96s
12:	learn: 0.4841772	total: 4.79s	remaining: 2.58s
13:	learn: 0.4568366	total: 5.15s	remaining: 2.21s
14:	learn: 0.4308819	total: 5.53s	remaining: 1.84s
15:	learn: 0.4079105	total: 5.89s	remaining: 1.47s
16:	learn: 0.3851139	total: 6.25s	remaining: 1.1s
17:	learn: 0.3621163	total: 6.62s	remaining: 736ms
18:	learn: 0.3441793	total: 6.99s	remaining: 368ms
19:	learn: 0.3272396	total: 7.37s	remainin

In [54]:
model2.save_model("output/model2.cbm")